# Project Kanto — YOLOv8n-cls training on Colab A100

End-to-end iNat21-Mini species classifier training on a single Colab A100.

**Open this notebook in Colab:**
https://colab.research.google.com/github/sharma-aryan-1/Project-Kanto/blob/main/ml_pipeline/notebooks/colab_train.ipynb

**Before you start, in Colab UI:** `Runtime → Change runtime type → A100 GPU` (Colab Pro+ required).

**The cells, in order, do:**
1. Confirm A100 + CUDA torch.
2. Mount Google Drive (persistent weights).
3. Clone the repo into `/content/Project-Kanto`.
4. Install Python deps.
5. Pull `train_mini.tar.gz` (44 GB) from S3 with MD5 verification.
6. Extract tar (~50 GB on /content) + build `taxonomy_map.csv`.
7. Build the YOLO classification dataset tree (500K symlinks).
8. Train YOLOv8n-cls (target ~2.5 h on A100 at batch=256, imgsz=224).
9. Copy `best.pt` and run artifacts to Drive so they survive a session disconnect.

Total wall-clock target: **~3 hours from cell 1 to a saved `best.pt` in Drive.**

## 1. Confirm A100 + CUDA torch

In [ ]:
!nvidia-smi --query-gpu=name,driver_version,memory.total,compute_cap --format=csv
import torch
print('torch:', torch.__version__, '| cuda available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No GPU detected; switch runtime to A100.'
name = torch.cuda.get_device_name(0)
print('GPU :', name)
if 'A100' not in name:
    print(f'\n  WARNING: expected A100 but got {name}. Training will still work but be slower.')

## 2. Mount Google Drive
Used only as a destination for final weights so they survive a Colab disconnect.
Dataset stays on `/content` (Drive I/O is too slow for 500K small files).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pathlib
DRIVE_OUT = pathlib.Path('/content/drive/MyDrive/project-kanto-runs')
DRIVE_OUT.mkdir(parents=True, exist_ok=True)
print('persistent run dir:', DRIVE_OUT)

## 3. Clone the repo

In [ ]:
%cd /content
![ -d Project-Kanto ] && rm -rf Project-Kanto
!git clone --depth 1 https://github.com/sharma-aryan-1/Project-Kanto.git
%cd Project-Kanto
!git log -1 --oneline

## 4. Install Python deps
Colab ships with CUDA torch + torchvision pre-installed, so we only add the project-specific extras (`ultralytics`, `albumentations`, etc.). We deliberately skip the lines in `requirements.txt` that would re-pin torch.

In [ ]:
!pip install -q ultralytics==8.4.46 albumentations==1.4.18 PyYAML==6.0.1 scikit-learn==1.4.1.post1 tqdm==4.66.5 requests==2.32.3
!python -c "import torch, torchvision, ultralytics; print('torch:', torch.__version__, '| torchvision:', torchvision.__version__, '| ultralytics:', ultralytics.__version__)"

## 5. Download iNat21-Mini archive (44 GB) from AWS S3 + verify MD5
S3 → Colab is typically ~200 MB/s, so this finishes in ~5 min and our `download_file` auto-pulls the canonical MD5 from S3 user-metadata and verifies it before extraction.

In [ ]:
%cd /content/Project-Kanto
# Run download-only first so we can fail fast if MD5 mismatches.
!python ml_pipeline/src/data_fetcher.py inat21-mini --skip-extract

## 6. Extract + build `taxonomy_map.csv`

In [ ]:
!python ml_pipeline/src/data_fetcher.py inat21-mini --skip-download --validate-paths

## 7. Build the YOLO classification dataset tree
On Linux (Colab) `os.symlink` works without elevation, so we use the spec'd default.

In [ ]:
!python ml_pipeline/src/data_prep.py --link-mode symlink --clean

## 8. Train YOLOv8n-cls

Defaults below are `--batch 384 --workers 6`, tuned for **system RAM**, not GPU VRAM. The A100's 40 GB VRAM can technically handle `batch=512 workers=12`, but on a standard Colab Pro A100 (~25 GB system RAM) the forked dataloader workers compound past the OOM ceiling and kill the runtime before epoch 1 finishes warmup.

**Per-epoch wall-clock at the defaults: ~2 min. Full 50-epoch run: ~75 min.**

If you are on the high-RAM A100 (Pro+), check `!free -g` first; if you have ≥50 GB available, you can push `--workers 8` for marginally better dataloader throughput. Don't go past 8 — copy-on-write breakage in forked Python workers makes the marginal RAM cost grow non-linearly.

`save_period=1` writes `last.pt` + `epochN.pt` after every epoch's validation pass, so any future disconnect costs at most one ~2-min epoch.

**Before running this cell**, paste this into the browser DevTools console (Cmd/Ctrl+Shift+J) on the Colab tab to suppress idle disconnects:

```javascript
setInterval(() => {
  const b = document.querySelector("colab-toolbar-button#connect");
  if (b) b.click();
}, 60_000);
```

In [ ]:
!free -g | head -2
!python ml_pipeline/src/train.py \
    --batch 384 \
    --imgsz 224 \
    --workers 6 \
    --save-period 1 \
    --name yolov8n-cls-inat21-mini-a100 \
    --exist-ok 2>&1 | tee /content/training.log

## 9. Copy run artifacts to Drive
This persists `weights/best.pt`, `weights/last.pt`, `results.csv`, plots, and `args.yaml` so you can disconnect the runtime and still have the trained model.

In [ ]:
import shutil, pathlib
RUN_DIR = pathlib.Path('/content/Project-Kanto/ml_pipeline/runs/yolov8n-cls-inat21-mini-a100')
DEST    = pathlib.Path('/content/drive/MyDrive/project-kanto-runs/yolov8n-cls-inat21-mini-a100')
if DEST.exists():
    shutil.rmtree(DEST)
shutil.copytree(RUN_DIR, DEST)
shutil.copy('/content/training.log', DEST / 'training.log')
print(f'\npersisted run to {DEST}')
for p in sorted(DEST.rglob('*')):
    if p.is_file():
        print(f'  {p.relative_to(DEST)}  ({p.stat().st_size/1e6:.2f} MB)')

## (Optional) Resume after a disconnect

If the Colab session disconnects mid-training:

1. Reconnect to a fresh runtime.
2. Run cells 1–4 (env, Drive mount, repo clone, deps) — these are required even if your data is gone.
3. Check whether `/content/Project-Kanto/ml_pipeline/data/processed/yolo_dataset/` still exists. If it doesn't, re-run cells 5–7 (~18 min).
4. Run the cell below.

**Important:** `--resume` only works if **at least one full epoch completed before the disconnect**, because Ultralytics only writes checkpoints between epochs (after validation). At `batch=512` that means ~90 s; if you disconnected sooner than that, no checkpoint exists anywhere and the cell below will detect this and start a fresh run instead.

In [ ]:
import pathlib, shutil, subprocess, sys

NAME      = 'yolov8n-cls-inat21-mini-a100'
DRIVE_RUN = pathlib.Path(f'/content/drive/MyDrive/project-kanto-runs/{NAME}')
LOCAL_RUN = pathlib.Path(f'/content/Project-Kanto/ml_pipeline/runs/{NAME}')
DATASET   = pathlib.Path('/content/Project-Kanto/ml_pipeline/data/processed/yolo_dataset')

if not DATASET.exists() or not (DATASET / 'train').exists():
    raise SystemExit(
        f'dataset not found at {DATASET}. Re-run cells 5, 6, 7 first to rebuild it (~18 min).'
    )

ckpt_in_drive = (DRIVE_RUN / 'weights' / 'last.pt').exists() if DRIVE_RUN.exists() else False
ckpt_local    = (LOCAL_RUN / 'weights' / 'last.pt').exists()

if ckpt_in_drive and not ckpt_local:
    LOCAL_RUN.parent.mkdir(parents=True, exist_ok=True)
    if LOCAL_RUN.exists():
        shutil.rmtree(LOCAL_RUN)
    shutil.copytree(DRIVE_RUN, LOCAL_RUN)
    print(f'[resume] restored prior run from {DRIVE_RUN}')
    ckpt_local = True

if ckpt_local:
    print(f'[resume] resuming from {LOCAL_RUN/"weights"/"last.pt"}')
    cmd = ['python', 'ml_pipeline/src/train.py', '--resume', '--name', NAME]
else:
    print(
        '[resume] no checkpoint found anywhere — the previous run never completed '
        'an epoch.\n          Falling back to a fresh training run with batch=512.'
    )
    cmd = [
        'python', 'ml_pipeline/src/train.py',
        '--batch', '384', '--imgsz', '224', '--workers', '6',
        '--save-period', '1', '--name', NAME, '--exist-ok',
    ]

with open('/content/training.log', 'w') as log:
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1, text=True)
    for line in proc.stdout:
        sys.stdout.write(line)
        log.write(line)
    proc.wait()